# Stage 00b — Descriptions CSV + VLM Figure Matching

**What this notebook does:**

1. Exports all *Brief Description of the Drawings* texts from the PatSeer Excel
   into a single `data/descriptions.csv` (replaces the old per-patent `.txt` files).

2. Assigns a figure label to every downloaded image using **Qwen2.5-VL-7B-Instruct (local GPU)** — the VLM reads the drawing sheet and returns structured JSON with bounding boxes and figure labels.

The matching logic is:
1. Run Qwen2.5-VL on the full image with a structured JSON prompt
2. Parse the returned `[{box, label}]` list; normalised coords 0–1000
3. Crop each bounding box and match to the detected FIG. label via regex
4. FAT files are always `_Fu` (copied whole, flagged for human review)
5. If VLM parsing fails: copy the full image as `_Fu` fallback

**Output naming:**

| Label source | Output filename | Meaning |
|---|---|---|
| VLM spatial match | `US…_F003A.png` | matched to FIG. 3A |
| Unmatched / FAT / error | `US…_Fu001.png` | needs review |

Output is written to `matched/<patent_id>/` — raw files are never modified.

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b  ←  YOU ARE HERE  (descriptions CSV + VLM matching → _F / _Fu labels)
 ↓
01   (optional human review for _Fu files)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|------|
| 1 | Load config, Excel index, imports |
| 2 | Save `data/descriptions.csv` |
| 3 | Initialise Qwen2.5-VL engine (once per session) |
| 4 | Run pipeline (respects `scan_limit` from config) |
| 5 | Review crops flagged for human inspection |

In [1]:
import torch
print("Is CUDA active inside Jupyter?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Connected GPU Model:", torch.cuda.get_device_name(0))

Is CUDA active inside Jupyter?: True
Connected GPU Model: NVIDIA GeForce RTX 2080 Ti


In [2]:
import sys
import os
import pandas as pd
import yaml
from pathlib import Path

# Resolve notebook parent workspace root directories safely
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

# Imports now find paths without triggering ModuleNotFound errors
from extractor import load_patseer_excel, save_descriptions_csv
from figure_matcher import build_engine, process_all_patents

# Load configuration settings
with open(project_root / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Unpack path configurations dynamically
base_path = cfg["paths"]["base"]
for path_key in ["raw_images", "text", "labels", "processed", "matched", "data"]:
    cfg["paths"][path_key] = cfg["paths"][path_key].replace("{{ paths.base }}", base_path)

print("Paths completely resolved from config.yaml. Ready to proceed.")

Paths completely resolved from config.yaml. Ready to proceed.


In [3]:
# 1. Load the original raw Excel export index tracking sheet
excel_path = Path(cfg["paths"]["patseer_excel"])
print(f"Reading PatSeer data from: {excel_path.name}...")
patseer_index = load_patseer_excel(excel_path)

# 2. Save the primary unified descriptions dataset framework
desc_csv_path = Path(cfg["paths"]["data"]) / "descriptions.csv"
save_descriptions_csv(patseer_index, desc_csv_path)

# 3. Read it back as a structured pandas DataFrame
df = pd.read_csv(desc_csv_path)
print(f"Stage 1 Complete: Created descriptions.csv with {len(df)} records.")


Reading PatSeer data from: 1639__dataset_08_06_26.xlsx...


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [4]:
# Load the 7B Vision Model onto your local GPU. It stays in memory for fast iterations.
engine = build_engine(cfg)

Loading local GPU Vision Model (4-bit): Qwen/Qwen2.5-VL-7B-Instruct...
Using model cache repository path: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/models/Qwen


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

VLM Engine loaded successfully on GPU within memory limits.


In [5]:
# Apply testing limiters safely based on development setup config profiles
scan_limit = cfg["extractor"].get("scan_limit", None)
test_df = df.head(scan_limit) if scan_limit else df

print(f"Beginning spatial image extraction loop over {len(test_df)} instances using Local GPU VLM...")
results_df = process_all_patents(test_df, cfg, engine)

# Save Stage 3 output map to disk
crops_csv_path = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
results_df.to_csv(crops_csv_path, index=False)
print(f"Stage 3 Complete: Crops written out and catalogued inside maps frame: {crops_csv_path}")

# Render operational summary distributions
if not results_df.empty:
    print("\n--- Pipeline Segmentation Summary Matrix ---")
    print(results_df.groupby("method")["output"].count())
else:
    print("Warning: Processing queue produced no outputs.")

Beginning spatial image extraction loop over 10 instances using Local GPU VLM...
  ⚠  No raw folder found for US2022267016A1 — skipping
  ⚠  No raw folder found for US11787551B1 — skipping
  ⚠  No raw folder found for US2022234745A1 — skipping
  ⚠  No raw folder found for US2015014475A1 — skipping


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  ✓ Successfully processed patent US2020148347A1
  ⚠  No raw folder found for US2023312117A1 — skipping
  ⚠  No raw folder found for US2022089279A1 — skipping
  ⚠  No raw folder found for US2019031361A1 — skipping
  ✓ Successfully processed patent US2020172235A1
  ⚠  No raw folder found for US2021403155A1 — skipping
Stage 3 Complete: Crops written out and catalogued inside maps frame: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/crops_mapping.csv

--- Pipeline Segmentation Summary Matrix ---
method
vlm_spatial_ocr    37
Name: output, dtype: int64


In [6]:
# Filter crops that failed automated label extraction or were marked unassigned
if results_df.empty or "needs_review" not in results_df.columns:
    print("No crops produced — nothing to review.")
else:
    flagged_df = results_df[results_df["needs_review"] == True]

    review_csv_path = Path(cfg["paths"]["data"]) / "needs_human_review.csv"
    flagged_df.to_csv(review_csv_path, index=False)
    print(f"Stage 4 Complete: Created {review_csv_path} with {len(flagged_df)} cases needing manual review.\n")

    if not flagged_df.empty:
        display(flagged_df[["patent_id", "original", "output", "label", "method"]].head(20))

Stage 4 Complete: Created /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/needs_human_review.csv with 0 cases needing manual review.

